# 文本分类实例

## Step1 导入相关包


In [ ]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification

## Step2 加载数据

In [ ]:
import pandas as pd

data = pd.read_csv("./data/ChnSentiCorp_htl_all.csv")
data.head()

In [ ]:
data = data.dropna()
data

## Step3 创建Dataset

In [ ]:
from torch.utils.data import Dataset

class MyDataset(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./data/ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()
    
    def __getitem__(self, index):
        return self.data.iloc[index]["review"],self.data.iloc[index]["label"]
    
    def __len__(self):
        return len(self.data)

In [ ]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])

## Step4 划分数据集

In [ ]:
import torch
from torch.utils.data import random_split

# 三划分
train_len = int(0.8 * len(dataset))
valid_len = int(0.1 * len(dataset))
test_len = len(dataset) - train_len - valid_len

trainset, validset, testset = random_split(
    dataset,
    [train_len, valid_len, test_len],
    generator=torch.Generator().manual_seed(42)
)
len(trainset),len(validset),len(testset)

In [ ]:
for i in range(10):
    print(trainset[i])

## Step5 创建Dataloader

In [ ]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")

def collate_func(batch):
    texts,labels = [],[]
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    inputs = tokenizer(texts,max_length=128,padding="max_length",truncation=True,return_tensors="pt", dtype=torch.long)
    inputs["labels"] = torch.tensor(labels)
    return inputs

In [ ]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset,batch_size=64,shuffle=True,collate_fn=collate_func)
validloader = DataLoader(validset,batch_size=128,shuffle=False,collate_fn=collate_func)
testloader = DataLoader(testset,batch_size=128,shuffle=False,collate_fn=collate_func)

In [ ]:
next(enumerate(trainloader))

## Step6 创建模型及优化器

In [ ]:
import torch.nn as nn
from transformers import AutoModel


class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("hfl/rbt3")
        
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, 2)    
        # 改进分类头（MLP）
        # self.classifier = nn.Sequential(
        #     nn.Linear(768, 256),
        #     nn.ReLU(),
        #     nn.Dropout(0.1),
        #     nn.Linear(256, 2)
        # )

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

        # CLS向量
        mean = outputs.last_hidden_state.mean(dim=1)

        # 加入 Dropout: 对池化后的全局特征进行随机失活
        mean = self.dropout(mean)

        logits = self.classifier(mean)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
from torch.optim import Adam


model = MyModel().cuda()
optimizer = Adam(model.parameters(), lr=1e-6)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=21)

## Step7 训练与验证

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def evaluate(model, validloader):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    preds = []
    labels = []

    with torch.inference_mode():
        for batch in validloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            output = model(**batch)
            loss = output['loss']
            logits = output['logits']

            total_loss += loss.item()

            pred = torch.argmax(logits, dim=-1)

            preds.extend(pred.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

            correct += (pred == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    acc = correct / total
    avg_loss = total_loss / len(validloader)

    return acc, avg_loss, preds, labels

def train(model, trainloader, validloader, optimizer, scheduler, epochs=3, log_step=100):
    global_step = 0

    train_losses = []
    valid_losses = []
    train_accs = []
    valid_accs = []

    best_acc = 0

    for ep in range(epochs):
        model.train()
        total_loss = 0
        correct_train = 0
        total_train = 0

        for batch in trainloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()

            output = model(**batch)
            loss = output['loss']

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            # 训练集 Accuracy
            pred = torch.argmax(output['logits'], dim=-1)
            correct_train += (pred == batch["labels"]).sum().item()
            total_train += batch["labels"].size(0)

            if global_step % log_step == 0:
                print(f"Epoch: {ep} | Step: {global_step} | Loss: {loss.item():.4f}")

            global_step += 1

        
        scheduler.step()
        train_acc = correct_train / total_train
        train_loss = total_loss / len(trainloader)

        # 验证
        val_acc, val_loss, preds, labels = evaluate(model, validloader)

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        valid_losses.append(val_loss)
        valid_accs.append(val_acc)

        print(f"\nEpoch {ep}:")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Acc: {train_acc:.4f}")
        print(f"Valid Loss: {val_loss:.4f}")
        print(f"Valid Acc: {val_acc:.4f}\n")

        # 保存最优模型
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_model.pt")
            print("✅ Best model saved!")

    return train_losses, train_accs, valid_losses, valid_accs, preds, labels

def plot_curves(train_losses, train_accs, valid_losses, valid_accs):
    epochs = range(len(train_losses))

    # Loss
    plt.figure()
    plt.plot(epochs, train_losses, label="Train Loss")
    plt.plot(epochs, valid_losses, label="Valid Loss")
    plt.legend()
    plt.title("Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.ylim(0, max(max(train_losses), max(valid_losses)) * 1.2)
    plt.grid(True, alpha=0.3)
    plt.show()

    # Accuracy
    plt.figure()
    plt.plot(epochs, train_accs, label="Train Accuracy")
    plt.plot(epochs, valid_accs, label="Valid Accuracy")
    plt.legend()
    plt.title("Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_confusion_matrix(labels, preds):
    cm = confusion_matrix(labels, preds)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

train_losses, train_accs, valid_losses, valid_accs, preds, labels = train(
    model,
    trainloader,
    validloader,
    optimizer,
    scheduler,
    epochs=21
)

plot_curves(train_losses, train_accs, valid_losses, valid_accs)

plot_confusion_matrix(labels, preds)

In [ ]:
import numpy as np


def evaluate():
    model.eval()
    acc_num = 0
    preds, labels = [], []
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k,v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output['logits'],dim=-1)
            preds.extend(pred.cpu().tolist())
            labels.extend(batch["labels"].cpu().tolist())
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    
    print(np.mean(preds), np.mean(labels))
    # print(confusion_matrix(labels, preds))
    return acc_num / len(validset)

def train(epoch=3,log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k,v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            output['loss'].backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep},global_step: {global_step},loss: {output['loss'].item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep : {ep}, acc : {acc}")

In [ ]:
# 不训练，直接evaluate
acc = evaluate()
print("微调前准确率:", acc)

In [ ]:
train()

In [ ]:
# 训练后，再次evaluate
acc = evaluate()
print("微调后准确率:", acc)

In [ ]:
import numpy as np


def final_test():
    model.eval()
    acc_num = 0
    preds, labels = [], []
    with torch.inference_mode():
        for batch in testloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k,v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output['logits'],dim=-1)
            preds.extend(pred.cpu().tolist())
            labels.extend(batch["labels"].cpu().tolist())
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    
    print(np.mean(preds), np.mean(labels))
    # print(confusion_matrix(labels, preds))
    return acc_num / len(testset)

final_acc = final_test()
print("最终测试集准确率:", final_acc)

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(model)

In [ ]:
model.save_pretrained("./rbt3_sentiment_model")

## Step9 模型预测

In [ ]:
sen = "我觉得这家酒店不错，饭很好吃"
id2_label = {0 : "差评!",1 :"好评!"}

model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen,return_tensors="pt")
    inputs = {k : v.cuda() for k,v in inputs.items()}
    logits = model(**inputs).logits
    pred = torch.argmax(logits,dim=-1)
    print(f"输入: {sen} \n 模型预测结果: {id2_label.get(pred.item())}")

In [ ]:
import gradio as gr

id2_label = {0: "差评!", 1: "好评!"}

def predict_sentiment(text):
    model.eval()
    with torch.inference_mode():
        inputs = tokenizer(text, return_tensors="pt", max_length=128, padding="max_length", truncation=True)
        inputs = {k: v.cuda() for k, v in inputs.items()}
        logits = model(**inputs).logits
        pred = torch.argmax(logits, dim=-1)
        label = id2_label.get(pred.item())
        confidence = torch.softmax(logits, dim=-1)[0][pred.item()].item()
        return f"{label} (置信度: {confidence:.2f})"

iface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=3, placeholder="请输入您要分析的酒店评论..."),
    outputs="text",
    title="酒店评论情感分析",
    description="输入一段酒店评论，模型将预测其情感倾向（好评或差评）。"
)

iface.launch()

In [ ]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification",model=model,tokenizer=tokenizer,device=0)

In [ ]:
pipe(sen)